# Biggest Polities: Direct HYDE × Cliopatria Computation

Overlay Cliopatria polity polygons directly on the HYDE 3.4 population grid
to compute population under each polity at each time step. No sampling needed.

For each HYDE time step (127 steps from 10,000 BCE to 2024 CE):
1. Find all Cliopatria polities active at that year
2. Rasterize each polygon onto the HYDE 2160×4320 grid
3. Sum population within each polygon

This gives population under each polity at each time step, from which we can
compute total person-years, total births, and population-over-time graphs.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import h5py
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds
from shapely.geometry import mapping
from tqdm import tqdm
import matplotlib.pyplot as plt

## 1. Load data

In [2]:
# Load HYDE population grid
ds = xr.open_dataset('../Raw_Data/HYDE34/NetCDF/population.nc', decode_times=False)
pop_all = ds['population']  # shape (127, 2160, 4320)
lats = pop_all.coords['lat'].values
lons = pop_all.coords['lon'].values

# Convert time to years (same formula as person.py)
hyde_years = (ds['time'].values // 365).astype(int) + 1

# Extend to 2025 by duplicating the 2024 grid (HYDE ends at 2024)
hyde_years = np.append(hyde_years, 2025)
print(f'HYDE: {len(hyde_years)} time steps, {hyde_years[0]} to {hyde_years[-1]} (2025 extrapolated from 2024)')
print(f'Grid: {pop_all.shape[1]}×{pop_all.shape[2]}')

# Load births grid (128 steps; first 127 align with population, 128th is 2025)
with h5py.File('../Processed_Data/births_array.h5', 'r') as f:
    births_all = f['data'][:128]
print(f'Births: {births_all.shape}')

# Rasterization transform (HYDE grid bounds)
TRANSFORM = from_bounds(
    lons.min() - 0.0833/2, lats.min() - 0.0833/2,
    lons.max() + 0.0833/2, lats.max() + 0.0833/2,
    4320, 2160
)

# Load land/sea mask and compute coastal cells
from scipy.ndimage import binary_dilation
landlake = np.loadtxt('../Raw_Data/HYDE34/general/landlake.asc', skiprows=6)
land = (landlake == 1)
sea = (landlake == -9999)
coastal = land & binary_dilation(sea)
print(f'Coastal cells: {coastal.sum():,}, interior land: {(land & ~coastal).sum():,}')

# Load Cliopatria
clio_raw = gpd.read_file('cliopatria_polities_only.geojson')
clio = clio_raw.copy()
clio['is_container'] = clio['Name'].str.startswith('(')

# Extend polities ending at 2024 to 2025 (they obviously still exist)
clio.loc[clio['ToYear'] == 2024, 'ToYear'] = 2025

n_polity = (~clio['is_container']).sum()
n_container = clio['is_container'].sum()
print(f'Cliopatria: {len(clio):,} total records ({n_polity:,} polities, {n_container:,} containers), {clio["Name"].nunique()} unique names')

HYDE: 128 time steps, -10000 to 2025 (2025 extrapolated from 2024)
Grid: 2160×4320
Births: (128, 2160, 4320)
Coastal cells: 65,196, interior land: 2,150,633


/opt/anaconda3/lib/python3.11/site-packages/pyogrio/core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Cliopatria: 15,690 total records (12,987 polities, 2,703 containers), 1618 unique names


/opt/anaconda3/lib/python3.11/site-packages/geopandas/array.py:348: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  self.crs = crs
/opt/anaconda3/lib/python3.11/site-packages/geopandas/geodataframe.py:464: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  level.crs = crs


## 2. Compute annual population and births for each Cliopatria polity

For each Cliopatria record (polygon with FromYear–ToYear):
1. Rasterize the polygon onto the HYDE grid (once per record)
2. At each HYDE time step within [FromYear, ToYear], sum population and births within the mask
3. Log-linear interpolate between HYDE steps to get annual values
4. When multiple records exist for the same polity name (different time periods or boundary changes), stitch them together

This gives an annual time series of population and births for every polity.

In [ ]:
# For each Cliopatria record: rasterize polygon, sum HYDE population AND births at
# relevant time steps, then log-linear interpolate to get annual values.

# Preload all HYDE grids into memory
print('Loading all HYDE grids...')
pop_grids = {}
birth_grids = {}
for ti, year in enumerate(hyde_years):
    if ti < len(pop_all.time):
        pop_grids[year] = pop_all.isel(time=ti).values
    else:
        pop_grids[year] = pop_grids[hyde_years[ti - 1]]
    if ti < len(births_all):
        birth_grids[year] = births_all[ti]
    else:
        birth_grids[year] = birth_grids[hyde_years[ti - 1]]

# World totals at each HYDE step
world_pop_hyde = {year: np.nansum(grid) for year, grid in pop_grids.items()}
world_births_hyde = {year: np.nansum(grid) for year, grid in birth_grids.items()}

def log_interp(hyde_yrs, hyde_vals, from_year, to_year):
    """Log-linear interpolate HYDE values to annual series for [from_year, to_year]."""
    hyde_vals_safe = np.where(hyde_vals > 0, hyde_vals, 1.0)
    all_zero = hyde_vals <= 0
    log_vals = np.log(hyde_vals_safe)
    exact = dict(zip(hyde_yrs.tolist(), hyde_vals.tolist()))
    
    annual = {}
    for year in range(from_year, to_year + 1):
        if year in exact:
            annual[year] = exact[year]
        elif year < hyde_yrs[0] or year > hyde_yrs[-1]:
            continue
        else:
            idx = np.searchsorted(hyde_yrs, year, side='right') - 1
            y0, y1 = hyde_yrs[idx], hyde_yrs[idx + 1]
            t = (year - y0) / (y1 - y0)
            if all_zero[idx] and all_zero[idx + 1]:
                annual[year] = 0.0
            else:
                annual[year] = np.exp(log_vals[idx] * (1 - t) + log_vals[idx + 1] * t)
    return annual

# Process each Cliopatria record
print(f'Processing {len(clio):,} Cliopatria records...')
record_results = []

all_hyde_sorted = sorted(hyde_years)

for _, row in tqdm(clio.iterrows(), total=len(clio), desc='Rasterizing'):
    name = row['Name']
    from_year = int(row['FromYear'])
    to_year = int(row['ToYear'])
    
    # Single rasterization with all_touched to capture coastal cells
    try:
        mask = geometry_mask([mapping(row.geometry)], out_shape=(2160, 4320),
                            transform=TRANSFORM, invert=True, all_touched=True)
    except Exception:
        continue
    
    if not mask.any():
        continue
    
    # Find HYDE years for interpolation
    active_hyde = [y for y in hyde_years if from_year <= y <= to_year]
    idx_before = np.searchsorted(all_hyde_sorted, from_year) - 1
    idx_after = np.searchsorted(all_hyde_sorted, to_year, side='right')
    if idx_before >= 0:
        active_hyde = [all_hyde_sorted[idx_before]] + active_hyde
    if idx_after < len(all_hyde_sorted):
        active_hyde = active_hyde + [all_hyde_sorted[idx_after]]
    active_hyde = sorted(set(active_hyde))
    
    if len(active_hyde) == 0:
        continue
    
    # Sum population and births within mask at each HYDE year
    hyde_pops = np.array([np.nansum(pop_grids[y][mask]) for y in active_hyde])
    hyde_births = np.array([np.nansum(birth_grids[y][mask]) for y in active_hyde])
    active_hyde = np.array(active_hyde)
    
    # Log-linear interpolate to every year in [from_year, to_year]
    pop_annual = log_interp(active_hyde, hyde_pops, from_year, to_year)
    births_annual = log_interp(active_hyde, hyde_births, from_year, to_year)
    
    for year in range(from_year, to_year + 1):
        pop = pop_annual.get(year)
        births = births_annual.get(year)
        if pop is not None:
            record_results.append((name, year, pop, births or 0.0))

print(f'{len(record_results):,} annual polity-year records')

df_annual = pd.DataFrame(record_results, columns=['polity', 'year', 'population', 'births'])
print(f'{df_annual["polity"].nunique()} polities')

# World annual series
world_pop_annual = log_interp(
    np.array(sorted(world_pop_hyde.keys())),
    np.array([world_pop_hyde[y] for y in sorted(world_pop_hyde.keys())]),
    int(hyde_years[0]), int(hyde_years[-1])
)
world_births_annual = log_interp(
    np.array(sorted(world_births_hyde.keys())),
    np.array([world_births_hyde[y] for y in sorted(world_births_hyde.keys())]),
    int(hyde_years[0]), int(hyde_years[-1])
)

## 3. Rankings: total person-years and total births by polity

In [ ]:
# Person-years, total births, and max population by polity
polity_py = df_annual.groupby('polity')['population'].sum().sort_values(ascending=False)
polity_births = df_annual.groupby('polity')['births'].sum().sort_values(ascending=False)
polity_max_pop = df_annual.groupby('polity')['population'].max().sort_values(ascending=False)

world_py = sum(world_pop_annual.values())
world_births = sum(world_births_annual.values())

from math import log10, floor
def sig2(x):
    if x == 0: return '0'
    d = -floor(log10(abs(x))) + 1
    r = round(x, d)
    if d <= 0: return f'{int(r):,}'
    return f'{r:,.{d}f}'

# Identify containers
container_names = set(clio[clio['is_container']]['Name'].unique())

print(f'Total person-years (world): {world_py/1e9:.0f} billion')
print(f'Total births (world): {world_births/1e9:.1f} billion')

def print_ranking(py_series, births_series, max_pop_series, title, name_filter, n=50):
    filtered = [(name, py) for name, py in py_series.items() if name_filter(name)]
    print(f'\n{"="*95}')
    print(f'{title}')
    print(f'{"="*95}')
    print(f'{"Rank":<5} {"Polity":<45} {"Pers-yrs (B)":>12} {"Births (M)":>10} {"Max pop (M)":>11}')
    print('-' * 95)
    for i, (name, py) in enumerate(filtered[:n], 1):
        births = births_series.get(name, 0)
        max_pop = max_pop_series.get(name, 0)
        print(f'{i:<5} {name:<45} {sig2(py/1e9):>12} {sig2(births/1e6):>10} {sig2(max_pop/1e6):>11}')

print_ranking(polity_py, polity_births, polity_max_pop, 'POLITIES (most specific)',
              lambda n: n not in container_names)
print_ranking(polity_py, polity_births, polity_max_pop, 'CONTAINERS (broadest umbrella)',
              lambda n: n in container_names, n=20)

## 4. Population over time: top 10 polities

In [ ]:
# Get top 10 non-container polities by person-years
top10 = [name for name, _ in polity_py.items() if name not in container_names][:10]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for polity in top10:
    pdata = df_annual[df_annual['polity'] == polity].set_index('year')['population'].sort_index()
    ax1.plot(pdata.index, pdata.values / 1e6, label=polity, linewidth=1.5)

ax1.set_ylabel('Population (millions)')
ax1.set_title('Population under largest polities')
ax1.legend(fontsize=8, loc='upper left')
ax1.set_xlim(-500, 2026)

# Share of world population
world_series = pd.Series(world_pop_annual).sort_index()

for polity in top10:
    pdata = df_annual[df_annual['polity'] == polity].set_index('year')['population'].sort_index()
    common_years = pdata.index.intersection(world_series.index)
    share = 100 * pdata.loc[common_years] / world_series.loc[common_years]
    ax2.plot(share.index, share.values, label=polity, linewidth=1.5)

ax2.set_ylabel('Share of world population (%)')
ax2.set_xlabel('Year')
ax2.legend(fontsize=8, loc='upper left')
ax2.set_xlim(-500, 2026)

plt.tight_layout()
plt.show()

## 5. Blog post cleanup

Merge name variants, rename confusing names, exclude non-polities, split ambiguous entries.

In [ ]:
POLITY_MERGES = {
    'Eastern Roman Empire': 'Byzantine Empire',
    'Kuomintang': 'Republic of China (mainland)',
    'Empire of China': 'Republic of China (mainland)',
    'Union of Myanmar': 'Myanmar',
    'Brazilian Republic': 'Brazil',
    'Federated Republic of Brazil': 'Brazil',
    'Republic of Brazil': 'Brazil',
    'Unitary State of Brazil': 'Brazil',
    'Federal Republic of Germany': 'West/Reunified Germany',
    'Federated Republic of Germany': 'West/Reunified Germany',
    'United Arab Republic': 'Egypt',
    'Arab Republic of Egypt': 'Egypt',
    'Mali Federation': 'Mali',
    'Iragi Republic': 'Iraq',
    'Republic of Iraq': 'Iraq',
}

POLITY_RENAMES = {
    'British Empire': 'East India Company',
    'Bourbon Kingdom of France': 'Restoration France',
    'Napoleonic Batavia Republic': 'Dutch Gold Coast',
}

POLITY_EXCLUDE = {
    'Holy Roman Empire Minor States',
}

def fmt_year(y):
    return f'{-y} BCE' if y < 0 else f'{y} CE'

def clean_polity_name(name, year):
    if name in POLITY_EXCLUDE:
        return None
    if name in POLITY_MERGES:
        name = POLITY_MERGES[name]
    if name in POLITY_RENAMES:
        name = POLITY_RENAMES[name]
    if name == 'Republic of China':
        return 'Republic of China (Taiwan)' if year >= 1950 else 'Republic of China (mainland)'
    if name == 'Burma':
        return 'Burma (Konbaung)' if year < 1948 else 'Myanmar'
    if name == 'Denmark-Norway' and year >= 1814:
        return 'Denmark'
    if name == 'Kingdom of Italy':
        return 'Kingdom of Italy (Lombard/Carolingian)' if year < 1000 else 'Kingdom of Italy (modern)'
    return name

# Apply cleaning to df_annual (non-containers only)
df_clean = df_annual[~df_annual['polity'].str.startswith('(')].copy()
df_clean['clean_name'] = df_clean.apply(lambda r: clean_polity_name(r['polity'], r['year']), axis=1)
df_clean = df_clean[df_clean['clean_name'].notna()]

# Aggregate
clean_py = df_clean.groupby('clean_name')['population'].sum().sort_values(ascending=False)
clean_births = df_clean.groupby('clean_name')['births'].sum().sort_values(ascending=False)
clean_max_pop = df_clean.groupby('clean_name')['population'].max().sort_values(ascending=False)
clean_years = df_clean.groupby('clean_name')['year'].agg(['min', 'max'])

print(f'{"Rank":<5} {"Polity":<45} {"Years":<20} {"Pers-yrs (B)":>12} {"Births (M)":>10} {"Max pop (M)":>11}')
print('-' * 108)
for i, (name, py) in enumerate(clean_py.head(300).items(), 1):
    births = clean_births.get(name, 0)
    max_pop = clean_max_pop.get(name, 0)
    yrs = clean_years.loc[name]
    yr_str = f'{fmt_year(int(yrs["min"]))}–{fmt_year(int(yrs["max"]))}'
    print(f'{i:<5} {name:<45} {yr_str:<20} {sig2(py/1e9):>12} {sig2(births/1e6):>10} {sig2(max_pop/1e6):>11}')

print(f'\n{len(clean_py)} polities after cleanup')

## 6. Super-polity groups

Broader entities: manually defined groups + Cliopatria containers.

In [ ]:
SUPER_POLITIES = {
    'Imperial China (canonical succession)': [
        'Qin Dynasty', 'Han Dynasty', 'Xin Dynasty',
        'Western Jin',
        'Sui Dynasty', 'Tang Dynasty',
        'Northern Song', 'Southern Song',
        'Yuan Dynasty', 'Ming Dynasty', 'Qing Dynasty',
    ],
    'Russia (Muscovy onward)': [
        'Grand Principality of Moscow', 'Tsardom of Russia', 'Russian Empire',
        'Russian Republic',
        'Union of Soviet Socialist Republics', 'Russian Federation',
    ],
    'Caliphate (Rashidun–Abbasid)': [
        'Rashidun Caliphate', 'Umayyad Caliphate', 'Abbasid Caliphate',
    ],
    'Roman state': [
        'Roman Republic', 'Roman Empire', 'Western Roman Empire',
        'Eastern Roman Empire', 'Byzantine Empire',
        'Nicaean Empire', 'Empire of Trebizond',
    ],
    'Genghisid domains': [
        'Mongol Empire',
        'Yuan Dynasty', 'Northern Yuan',
        'Golden Horde', 'Chagatai Khanate', 'Eastern Chagatai Khanate', 'Ilkhanate',
        'Crimean Khanate', 'Kazakh Khanate', 'Khanate of Kazan',
        'Astrakhan Khanate', 'Khanate of Sibir', 'Qasim Khanate',
        'Khanate of Bukhara', 'Khanate of Khiva', 'Mongol Khanate',
    ],
    'Habsburg domains': [
        'House of Habsburg', 'Habsburg Monarchy',
        'Austrian Empire', 'Austria-Hungary',
        ('Kingdom of Bohemia', 1526, 9999),
        ('Kingdom of Hungary', 1526, 1918),
        'Principality of Transylvania',
        ('Kingdom of Spain', 1516, 1700),
        ('Spanish Empire', 1516, 1700),
        'Iberian Union', 'Pro-Habsburg Spain',
    ],
    'Bourbon domains': [
        ('Kingdom of France', 1589, 1792),
        'Bourbon Kingdom of France',
        ('New France', 1589, 1792),
        ('French India', 1589, 1792),
        ('Kingdom of Spain', 1700, 9999),
        ('Spanish Empire', 1700, 1879),
        'Kingdom of the Two Sicilies',
        ('Kingdom of Naples (Napoleonic)', 1734, 1806),
        'Duchy of Parma and Piacenza',
    ],
    'Japan (under the Emperor)': [
        'Yamato', 'Kamakura Shogunate', 'Ashikaga Shogunate',
        'Tokugawa Shogunate', 'Empire of Japan', 'Japan',
    ],
    'British Empire (~1600–1997)': [
        # Metropole from 1600
        ('Kingdom of England', 1600, 9999),
        'Kingdom of Great Britain',
        # Colonial
        'British Colonial Empire', 'British Empire',  # EIC India
        'British Raj', 'British Africa', 'British Cape Colony',
        'British Ceylon', 'British East Africa', 'British Overseas Territories',
    ],
    'French Empire (~1600–1960)': [
        # Metropole from 1600
        ('Kingdom of France', 1600, 1791),
        'French First Republic', 'First French Empire',
        'Bourbon Kingdom of France', 'French Second Republic',
        'Second French Empire', 'French Third Republic',
        'Vichy France', 'Free French', 'French Fourth Republic',
        # Colonial
        'New France', 'French Africa', 'French Algiers',
        'French Equatorial Africa', 'French India', 'French Indochina',
        'French Colonial Vietnam', 'French Mandate for Syria and Lebanon',
        'French Louisiana', 'French Colony of Guiana', 'France Antarctique',
    ],
    'Spanish Empire (1516–1898)': [
        'Kingdom of Spain',
        ('Spanish Empire', None, 1898),
        'Iberian Union',
    ],
    'Portuguese Empire (~1500–1975)': [
        ('Kingdom of Portugal', 1500, 9999),
        'Portuguese Republic',
        ('Portugal', None, 1975),
        'Portuguese Empire', 'Portuguese Africa', 'Portuguese Colonies',
        'Portuguese Ceylon',
        ('Empire of Brazil', None, 1822),  # Brazil only while Portuguese colony
    ],
    'Dutch Empire (~1600–1949)': [
        'Dutch Republic', 'Batavian Republic',
        'Sovereign Principality of the United Netherlands',
        ('Netherlands', None, 1949),
        'Dutch East Indies', 'Dutch Cape Coast', 'Dutch Ceylon',
        'New Netherland', 'Napoleonic Batavia Republic',
        ('Netherlands Antilles', None, 1949),
    ],
    'Italian Empire (1861–1945)': [
        ('Kingdom of Italy', 1861, 1946),
        'Italian Africa',
    ],
    'German Empire (1871–1919)': [
        'German Empire', 'German Africa',
    ],
    'Belgian Empire': [
        'Kingdom of Belgium', 'Belgian Congo',
    ],
}

CONTAINER_GROUPS = [
    '(Delhi Sultanate)',
    '(Holy Roman Empire)',
]

def count_super_polity_direct(members, df):
    """Filter df_annual rows matching a super-polity group."""
    masks = []
    for member in members:
        if isinstance(member, tuple):
            name, start, end = member
            start = start if start is not None else -99999
            end = end if end is not None else 99999
            masks.append((df['polity'] == name) & (df['year'] >= start) & (df['year'] <= end))
        else:
            masks.append(df['polity'] == member)
    combined = masks[0]
    for m in masks[1:]:
        combined = combined | m
    return df[combined]

# Compute for all groups
group_results = []

for group_name, members in SUPER_POLITIES.items():
    sub = count_super_polity_direct(members, df_annual)
    py = sub['population'].sum()
    births = sub['births'].sum()
    max_pop = sub.groupby('year')['population'].sum().max() if len(sub) > 0 else 0
    min_year = sub['year'].min() if len(sub) > 0 else 0
    max_year = sub['year'].max() if len(sub) > 0 else 0
    group_results.append((group_name, py, births, max_pop, min_year, max_year))

for container_name in CONTAINER_GROUPS:
    sub = df_annual[df_annual['polity'] == container_name]
    py = sub['population'].sum()
    births = sub['births'].sum()
    max_pop = sub['population'].max() if len(sub) > 0 else 0
    min_year = sub['year'].min() if len(sub) > 0 else 0
    max_year = sub['year'].max() if len(sub) > 0 else 0
    group_results.append((container_name, py, births, max_pop, min_year, max_year))

group_results.sort(key=lambda x: -x[1])

print(f'{"Rank":<5} {"Super-polity":<45} {"Years":<18} {"Pers-yrs (B)":>12} {"Births (M)":>10} {"Max pop (M)":>11}')
print('-' * 105)
for i, (name, py, births, max_pop, yr_min, yr_max) in enumerate(group_results, 1):
    yr_str = f'{fmt_year(int(yr_min))}–{fmt_year(int(yr_max))}'
    print(f'{i:<5} {name:<45} {yr_str:<18} {sig2(py/1e9):>12} {sig2(births/1e6):>10} {sig2(max_pop/1e6):>11}')

## 7. Metropole vs colonial populations

In [ ]:
# For colonial empires, split into metropole vs colonial holdings
# Metropole dates are clipped to the empire's active period.

EMPIRE_SPLITS = {
    'British Empire (~1600–1997)': {
        'metropole': [('Kingdom of England', 1600, 1708), ('Kingdom of Great Britain', None, 1997)],
        'colonial': ['British Colonial Empire', 'British Empire', 'British Raj',
                     'British Africa', 'British Cape Colony', 'British Ceylon',
                     'British East Africa', 'British Overseas Territories'],
    },
    'French Empire (~1600–1960)': {
        'metropole': [
            ('Kingdom of France', 1600, 1791),
            'French First Republic', 'First French Empire',
            'Bourbon Kingdom of France', 'French Second Republic',
            'Second French Empire', 'French Third Republic',
            'Vichy France', 'Free French', 'French Fourth Republic',
        ],
        'colonial': [
            'New France', 'French Africa', 'French Algiers',
            'French Equatorial Africa', 'French India', 'French Indochina',
            'French Colonial Vietnam', 'French Mandate for Syria and Lebanon',
            'French Louisiana', 'French Colony of Guiana', 'France Antarctique',
        ],
    },
    'Spanish Empire (1516–1898)': {
        'metropole': [('Kingdom of Spain', None, 1898)],
        'colonial': [('Spanish Empire', None, 1898), 'Iberian Union'],
    },
    'Portuguese Empire (~1500–1975)': {
        'metropole': [('Kingdom of Portugal', 1500, 1911), ('Portuguese Republic', None, 1975),
                      ('Portugal', None, 1975), ('Estado Novo', None, 1975)],
        'colonial': ['Portuguese Empire', 'Portuguese Africa', 'Portuguese Colonies',
                     'Portuguese Ceylon', ('Empire of Brazil', None, 1822)],
        # Note: Portuguese Republic and Estado Novo include both metropole and colonies
        # in a single polygon, so the metropole/colonial split is not clean for 1911-1975.
    },
    'Dutch Empire (~1600–1949)': {
        'metropole': ['Dutch Republic', 'Batavian Republic',
                      'Sovereign Principality of the United Netherlands',
                      ('Netherlands', None, 1949)],
        'colonial': ['Dutch East Indies', 'Dutch Cape Coast', 'Dutch Ceylon',
                     'New Netherland', 'Napoleonic Batavia Republic',
                     ('Netherlands Antilles', None, 1949)],
    },
    'Italian Empire (1861–1945)': {
        'metropole': [('Kingdom of Italy', 1861, 1945)],
        'colonial': ['Italian Africa'],
    },
    'German Empire (1871–1919)': {
        'metropole': ['German Empire'],
        'colonial': ['German Africa'],
    },
    'Belgian Empire': {
        'metropole': [('Kingdom of Belgium', 1890, 1962)],
        'colonial': ['Belgian Congo'],
    },
}

def compute_split(members, df):
    sub = count_super_polity_direct(members, df)
    if len(sub) == 0:
        return 0, 0, pd.Series(dtype=float)
    py = sub['population'].sum()
    births = sub['births'].sum()
    annual_pop = sub.groupby('year')['population'].sum()
    return py, births, annual_pop

for empire_name, split in EMPIRE_SPLITS.items():
    m_py, m_births, m_annual = compute_split(split['metropole'], df_annual)
    c_py, c_births, c_annual = compute_split(split['colonial'], df_annual)
    t_py = m_py + c_py
    t_births = m_births + c_births
    
    # Peak year: max of combined population
    combined_annual = m_annual.add(c_annual, fill_value=0)
    if len(combined_annual) > 0:
        peak_year = int(combined_annual.idxmax())
        t_max = combined_annual[peak_year]
        m_at_peak = m_annual.get(peak_year, 0)
        c_at_peak = c_annual.get(peak_year, 0)
        pct_colonial_peak = 100 * c_at_peak / t_max if t_max > 0 else 0
    else:
        peak_year = t_max = m_at_peak = c_at_peak = pct_colonial_peak = 0
    
    pct_colonial_py = 100 * c_py / t_py if t_py > 0 else 0
    pct_colonial_births = 100 * c_births / t_births if t_births > 0 else 0
    
    print(f'=== {empire_name} ===')
    print(f'  {"":12} {"Pers-yrs (B)":>12} {"Births (M)":>10}')
    print(f'  {"Total":<12} {sig2(t_py/1e9):>12} {sig2(t_births/1e6):>10}')
    print(f'  {"Metropole":<12} {sig2(m_py/1e9):>12} {sig2(m_births/1e6):>10}')
    print(f'  {"Colonial":<12} {sig2(c_py/1e9):>12} {sig2(c_births/1e6):>10}')
    print(f'  {"% colonial":<12} {pct_colonial_py:>11.1f}% {pct_colonial_births:>9.1f}%')
    print(f'  Peak population: {sig2(t_max/1e6)}M in {peak_year} (metro {sig2(m_at_peak/1e6)}M + colonial {sig2(c_at_peak/1e6)}M, {pct_colonial_peak:.0f}% colonial)')
    print()